In [ ]:
%run "./00_CC_Mapping_Setup.ipynb" 

In [ ]:
%sql
/* ===================================================================================
   METRIC: TP05 - TP Gov't Int
   
   DATA SOURCES:
   1. Master AU List: hive_metastore.ra_adido_2025.fy25_list_of_units
   2. TP Unit Mapping: hive_metastore.ra_adido_2025.third_party_unit_mapping
   3. Third Party Data: hive_metastore.ra_adido_2025.abac_request_paul_v3
   
   BUSINESS QUESTION: 
   Number of third party relationships or engagements which required 1. the arrangement 
   of government issued approval, license and/or permits, etc. OR 2. the third party 
   to apply or interact with a PO on behalf of TD?
   
   LOGIC & ARCHITECTURE SUMMARY:
   1. MASTER AU ANCHOR: Uses the Master AU list as the absolute key for the output. 
      Only AUs in this filtered list (Canada, HK, Barbados + US_OR_CANADA = 'CANADA') 
      will appear in the final report.
   2. RISK FLAG LOGIC: Flags engagements as high risk if KPI_Number = '8.2' AND 
      final_rating = 'YES'.
   3. COMMA EXCEPTION HANDLING: Protects unit names containing commas (e.g., 'TECE - 
      Strategy, Change...') using the '[COMMA]' placeholder swap to prevent 
      improper splitting during the explosion phase.
   4. FLATTEN LOBs: Uses LATERAL VIEW explode(split(...)) to expand comma-separated 
      LOB lists. Restores '[COMMA]' to a real ',' after the split.
   5. SAFE MAPPING: Maps expanded LOBs to TPRM_Units using standard LIKE syntax.
   6. DUAL-BRIDGE JOIN: Matches on either the String Name OR the Numeric BusinessID 
      provided in the mapping table's Assessable_Unit_ID column.
   7. FINAL OUTPUT: Strict 6-column structure, counting the distinct numerical 
      engagements mapped to the AU and converting NULL sums to '0'.
=================================================================================== */

WITH Master_AUs AS (
    -- 1. Grab Master List data: This is our strict anchor for the final output
    SELECT 
        TRIM(CAST(BusinessID AS STRING)) AS BusinessID,
        TRIM(Business_Operational_Unit_Name) AS AU_Name,
        TRIM(BusinessSegment) AS Business_Segment,
        'Yes' AS In_ABAC_AU_List
    FROM hive_metastore.ra_adido_2025.fy25_list_of_units
    WHERE BusinessID IS NOT NULL
      AND UPPER(TRIM(Jurisdiction)) IN ('CANADA', 'HONG KONG', 'BARBADOS')
      AND UPPER(TRIM(US_OR_CANADA)) = 'CANADA'
),

Mapped_AUs AS (
    -- 2. Grab valid TPRM Mapping Strings (Accommodates mixed ID/Name format)
    SELECT DISTINCT 
        TRIM(Assessable_Unit_ID) AS Mapping_ID_Or_Name,
        TRIM(TPRM_Units) AS TPRM_Units
    FROM hive_metastore.ra_adido_2025.third_party_unit_mapping
    WHERE Assessable_Unit_ID IS NOT NULL
      AND NULLIF(TRIM(TPRM_Units), '') IS NOT NULL
),

Third_Party_Risk_Data AS (
    -- 3. Extract columns and apply Comma Protection Dictionary
    SELECT 
        EngagementNumber,
        KPI_Number,
        final_rating,
        
        -- EXCEPTION DICTIONARY: Chain replaces to shield internal commas from split()
        REPLACE(REPLACE(REPLACE(REPLACE(REPLACE(REPLACE(owning_lob, 
            'CAD PB - RESL, Everyday Banking, Savings & Investing', 'CAD PB - RESL[COMMA] Everyday Banking[COMMA] Savings & Investing'),
            'CAD PB - Card Payments, Loyalty, & Personal Lending', 'CAD PB - Card Payments[COMMA] Loyalty[COMMA] & Personal Lending'),
            'Transformation, Enablement, & Customer Experience', 'Transformation[COMMA] Enablement[COMMA] & Customer Experience'),
            'TECE - Strategy, Change, & Operational Excellence', 'TECE - Strategy[COMMA] Change[COMMA] & Operational Excellence'),
            'TECE - Enterprise Digital Strategy, Innovation, & Payments', 'TECE - Enterprise Digital Strategy[COMMA] Innovation[COMMA] & Payments'),
            'P&T - Integrations, Data and Payments', 'P&T - Integrations[COMMA] Data and Payments') AS safe_owning_lob,
            
        REPLACE(REPLACE(REPLACE(REPLACE(REPLACE(REPLACE(lob_using, 
            'CAD PB - RESL, Everyday Banking, Savings & Investing', 'CAD PB - RESL[COMMA] Everyday Banking[COMMA] Savings & Investing'),
            'CAD PB - Card Payments, Loyalty, & Personal Lending', 'CAD PB - Card Payments[COMMA] Loyalty[COMMA] & Personal Lending'),
            'Transformation, Enablement, & Customer Experience', 'Transformation[COMMA] Enablement[COMMA] & Customer Experience'),
            'TECE - Strategy, Change, & Operational Excellence', 'TECE - Strategy[COMMA] Change[COMMA] & Operational Excellence'),
            'TECE - Enterprise Digital Strategy, Innovation, & Payments', 'TECE - Enterprise Digital Strategy[COMMA] Innovation[COMMA] & Payments'),
            'P&T - Integrations, Data and Payments', 'P&T - Integrations[COMMA] Data and Payments') AS safe_lob_using
            
    FROM hive_metastore.ra_adido_2025.abac_request_paul_v3
    WHERE EngagementNumber IS NOT NULL
),

Flagged_Engagements AS (
    -- 4. Filter the TP data strictly based on KPI 8.2 and final_rating = YES
    SELECT DISTINCT
        EngagementNumber,
        safe_owning_lob,
        safe_lob_using
    FROM Third_Party_Risk_Data
    WHERE TRIM(KPI_Number) = '8.2' 
      AND UPPER(TRIM(final_rating)) = 'YES'
),

Flattened_LOBs AS (
    -- 5. Explode comma-separated strings into individual rows and restore commas
    SELECT 
        EngagementNumber, 
        REPLACE(TRIM(exploded_lob), '[COMMA]', ',') AS Expanded_LOB
    FROM Flagged_Engagements
    LATERAL VIEW explode(split(concat_ws(',', safe_owning_lob, safe_lob_using), ',')) AS exploded_lob
    WHERE exploded_lob IS NOT NULL AND TRIM(exploded_lob) != ''
),

Engagements_By_AU AS (
    -- 6. Map expanded rows to AU using Dual-Bridge and count matches
    SELECT 
        mast.BusinessID,
        -- DEDUPLICATION: Count distinct engagements to avoid double-counting expanded rows
        COUNT(DISTINCT f.EngagementNumber) AS Match_Count
    FROM Flattened_LOBs f
    INNER JOIN Mapped_AUs map
        ON UPPER(f.Expanded_LOB) LIKE '%' || UPPER(map.TPRM_Units) || '%'
    
    -- DUAL-BRIDGE JOIN: Supports mapping table's mix of Numeric ID or String Name
    INNER JOIN Master_AUs mast
        ON UPPER(TRIM(map.Mapping_ID_Or_Name)) = UPPER(TRIM(mast.AU_Name))
        OR TRIM(map.Mapping_ID_Or_Name) = TRIM(mast.BusinessID)
        
    GROUP BY mast.BusinessID
)

-- 7. Final Output: Strictly structured per Master Anchor
SELECT 
    a.BusinessID,                          
    a.AU_Name,                             
    a.Business_Segment,
    'TP05' AS QuestionID,               
    COALESCE(CAST(e.Match_Count AS STRING), '0') AS Response,
    a.In_ABAC_AU_List
    
FROM Master_AUs a
LEFT JOIN Engagements_By_AU e 
    ON a.BusinessID = e.BusinessID
ORDER BY a.BusinessID;

In [ ]:
%sql
/* ===================================================================================
   DEBUG TABLE: TP05 - Production Logic With AU-Level Evidence
   PURPOSE: Starts from the exact production join path, then adds compact evidence
   columns so each AU result can be reviewed without materializing the full row set.
=================================================================================== */

WITH Master_AUs AS (
    SELECT 
        TRIM(CAST(BusinessID AS STRING)) AS BusinessID,
        TRIM(Business_Operational_Unit_Name) AS AU_Name,
        TRIM(BusinessSegment) AS Business_Segment,
        'Yes' AS In_ABAC_AU_List
    FROM hive_metastore.ra_adido_2025.fy25_list_of_units
    WHERE BusinessID IS NOT NULL
      AND UPPER(TRIM(Jurisdiction)) IN ('CANADA', 'HONG KONG', 'BARBADOS')
      AND UPPER(TRIM(US_OR_CANADA)) = 'CANADA'
),

Mapped_AUs AS (
    SELECT DISTINCT 
        TRIM(Assessable_Unit_ID) AS Mapping_ID_Or_Name,
        TRIM(TPRM_Units) AS TPRM_Units
    FROM hive_metastore.ra_adido_2025.third_party_unit_mapping
    WHERE Assessable_Unit_ID IS NOT NULL
      AND NULLIF(TRIM(TPRM_Units), '') IS NOT NULL
),

Third_Party_Risk_Data AS (
    SELECT 
        EngagementNumber,
        KPI_Number,
        final_rating,
        REPLACE(REPLACE(REPLACE(REPLACE(REPLACE(REPLACE(owning_lob, 
            'CAD PB - RESL, Everyday Banking, Savings & Investing', 'CAD PB - RESL[COMMA] Everyday Banking[COMMA] Savings & Investing'),
            'CAD PB - Card Payments, Loyalty, & Personal Lending', 'CAD PB - Card Payments[COMMA] Loyalty[COMMA] & Personal Lending'),
            'Transformation, Enablement, & Customer Experience', 'Transformation[COMMA] Enablement[COMMA] & Customer Experience'),
            'TECE - Strategy, Change, & Operational Excellence', 'TECE - Strategy[COMMA] Change[COMMA] & Operational Excellence'),
            'TECE - Enterprise Digital Strategy, Innovation, & Payments', 'TECE - Enterprise Digital Strategy[COMMA] Innovation[COMMA] & Payments'),
            'P&T - Integrations, Data and Payments', 'P&T - Integrations[COMMA] Data and Payments') AS safe_owning_lob,
        REPLACE(REPLACE(REPLACE(REPLACE(REPLACE(REPLACE(lob_using, 
            'CAD PB - RESL, Everyday Banking, Savings & Investing', 'CAD PB - RESL[COMMA] Everyday Banking[COMMA] Savings & Investing'),
            'CAD PB - Card Payments, Loyalty, & Personal Lending', 'CAD PB - Card Payments[COMMA] Loyalty[COMMA] & Personal Lending'),
            'Transformation, Enablement, & Customer Experience', 'Transformation[COMMA] Enablement[COMMA] & Customer Experience'),
            'TECE - Strategy, Change, & Operational Excellence', 'TECE - Strategy[COMMA] Change[COMMA] & Operational Excellence'),
            'TECE - Enterprise Digital Strategy, Innovation, & Payments', 'TECE - Enterprise Digital Strategy[COMMA] Innovation[COMMA] & Payments'),
            'P&T - Integrations, Data and Payments', 'P&T - Integrations[COMMA] Data and Payments') AS safe_lob_using
    FROM hive_metastore.ra_adido_2025.abac_request_paul_v3
    WHERE EngagementNumber IS NOT NULL
),

Flagged_Engagements AS (
    SELECT DISTINCT
        EngagementNumber,
        safe_owning_lob,
        safe_lob_using
    FROM Third_Party_Risk_Data
    WHERE TRIM(KPI_Number) = '8.2' 
      AND UPPER(TRIM(final_rating)) = 'YES'
),

Flattened_LOBs AS (
    SELECT 
        EngagementNumber, 
        REPLACE(TRIM(exploded_lob), '[COMMA]', ',') AS Expanded_LOB
    FROM Flagged_Engagements
    LATERAL VIEW explode(split(concat_ws(',', safe_owning_lob, safe_lob_using), ',')) AS exploded_lob
    WHERE exploded_lob IS NOT NULL AND TRIM(exploded_lob) != ''
),


TP_Names AS (
    -- Lookup: Distinct EngagementNumber -> ThirdPartyName
    SELECT DISTINCT
        TRIM(CAST(EngagementNumber AS STRING)) AS EngagementNumber,
        TRIM(ThirdPartyName) AS ThirdPartyName
    FROM hive_metastore.ra_adido_2025.abac_request_paul_v3
    WHERE EngagementNumber IS NOT NULL
      AND ThirdPartyName IS NOT NULL
),

Resolved_Rows AS (
    SELECT DISTINCT
        mast.BusinessID,
        mast.AU_Name,
        mast.Business_Segment,
        f.EngagementNumber,
        f.Expanded_LOB,
        map.TPRM_Units,
        map.Mapping_ID_Or_Name,
        CASE 
            WHEN TRIM(map.Mapping_ID_Or_Name) = TRIM(mast.BusinessID) THEN 'Matched on AU ID'
            WHEN UPPER(TRIM(map.Mapping_ID_Or_Name)) = UPPER(TRIM(mast.AU_Name)) THEN 'Matched on AU Name'
            ELSE 'Matched through dual bridge'
        END AS Bridge_Method,
        tp.ThirdPartyName
    FROM Flattened_LOBs f
    INNER JOIN Mapped_AUs map
        ON UPPER(f.Expanded_LOB) LIKE '%' || UPPER(map.TPRM_Units) || '%'
    INNER JOIN Master_AUs mast
        ON UPPER(TRIM(map.Mapping_ID_Or_Name)) = UPPER(TRIM(mast.AU_Name))
        OR TRIM(map.Mapping_ID_Or_Name) = TRIM(mast.BusinessID)
    LEFT JOIN TP_Names tp
        ON TRIM(CAST(f.EngagementNumber AS STRING)) = tp.EngagementNumber
),

Engagements_By_AU AS (
    SELECT 
        BusinessID,
        COUNT(DISTINCT EngagementNumber) AS Match_Count,
        COUNT(*) AS Resolved_Row_Count
    FROM Resolved_Rows
    GROUP BY BusinessID
),

Trace_By_AU AS (
    SELECT 
        BusinessID,
        CONCAT_WS(', ', SLICE(SORT_ARRAY(COLLECT_SET(TPRM_Units)), 1, 15)) AS Matched_TPRM_Units_Sample,
        CONCAT_WS(', ', SLICE(SORT_ARRAY(COLLECT_SET(Mapping_ID_Or_Name)), 1, 15)) AS Mapping_Bridge_Value_Sample,
        CONCAT_WS(', ', SLICE(SORT_ARRAY(COLLECT_SET(Expanded_LOB)), 1, 15)) AS Expanded_LOB_Sample,
        CONCAT_WS(', ', SLICE(SORT_ARRAY(COLLECT_SET(CAST(EngagementNumber AS STRING))), 1, 15)) AS Engagement_Sample,
        CONCAT_WS(', ', TRANSFORM(
            SORT_ARRAY(COLLECT_SET(NAMED_STRUCT('eng', CAST(EngagementNumber AS STRING), 'name', COALESCE(ThirdPartyName, '')))),
            x -> x.name
        )) AS Third_Party_Names,
        CONCAT_WS(', ', SORT_ARRAY(COLLECT_SET(Bridge_Method))) AS Bridge_Method_List
    FROM Resolved_Rows
    GROUP BY BusinessID
)

-- Output columns:
-- BusinessID: Master AU ID from the ABAC AU anchor list.
-- AU_Name: Master AU name tied to BusinessID.
-- Business_Segment: Business segment carried from the master AU list.
-- QuestionID: Questionnaire code for this debug output.
-- Response: Final distinct-engagement count returned for TP05.
-- Counted_Engagement_Count: Final number of distinct qualifying engagements.
-- Resolved_Row_Count: Number of resolved bridge rows before distinct-engagement deduplication.
-- Matched_TPRM_Units_Sample: Sample TPRM unit values that matched this AU.
-- Mapping_Bridge_Value_Sample: Sample mapping-table values used to bridge this AU.
-- Expanded_LOB_Sample: Sample flattened LOB values that matched the mapping table.
-- Engagement_Sample: Sample engagement numbers supporting the result.
-- Third_Party_Names: Third party vendor names corresponding to each engagement in Engagement_Sample, in matching positional order.
-- Bridge_Method_List: Sample of whether bridging happened by AU ID, AU name, or both.
-- Calculation_Formula: Plain-English explanation of how Response was produced.
-- In_ABAC_AU_List: Confirms the AU came from the standard ABAC AU list.
SELECT 
    a.BusinessID,
    a.AU_Name,
    a.Business_Segment,
    'TP05' AS QuestionID,
    COALESCE(CAST(e.Match_Count AS STRING), '0') AS Response,
    COALESCE(e.Match_Count, 0) AS Counted_Engagement_Count,
    COALESCE(e.Resolved_Row_Count, 0) AS Resolved_Row_Count,
    COALESCE(t.Matched_TPRM_Units_Sample, 'n/a') AS Matched_TPRM_Units_Sample,
    COALESCE(t.Mapping_Bridge_Value_Sample, 'n/a') AS Mapping_Bridge_Value_Sample,
    COALESCE(t.Expanded_LOB_Sample, 'n/a') AS Expanded_LOB_Sample,
    COALESCE(t.Engagement_Sample, 'n/a') AS Engagement_Sample,
    COALESCE(t.Third_Party_Names, 'n/a') AS Third_Party_Names,
    COALESCE(t.Bridge_Method_List, 'n/a') AS Bridge_Method_List,
    CASE 
        WHEN COALESCE(e.Match_Count, 0) = 0 THEN '0 because no KPI 8.2 = YES engagement bridged to this AU'
        ELSE CONCAT('Response = count(distinct EngagementNumber). Resolved rows before dedupe = ', CAST(e.Resolved_Row_Count AS STRING))
    END AS Calculation_Formula,
    a.In_ABAC_AU_List
FROM Master_AUs a
LEFT JOIN Engagements_By_AU e 
    ON a.BusinessID = e.BusinessID
LEFT JOIN Trace_By_AU t
    ON a.BusinessID = t.BusinessID
ORDER BY a.BusinessID;


In [ ]:
%sql
/* ===================================================================================
   DEBUG TABLE 2: TP05 - Sample Resolved Rows
   PURPOSE: Provides a capped sample of the exact resolved production rows so the
   evidence stays readable and avoids the heavier full-row debug path.
=================================================================================== */

WITH Master_AUs AS (
    SELECT 
        TRIM(CAST(BusinessID AS STRING)) AS BusinessID,
        TRIM(Business_Operational_Unit_Name) AS AU_Name,
        TRIM(BusinessSegment) AS Business_Segment,
        'Yes' AS In_ABAC_AU_List
    FROM hive_metastore.ra_adido_2025.fy25_list_of_units
    WHERE BusinessID IS NOT NULL
      AND UPPER(TRIM(Jurisdiction)) IN ('CANADA', 'HONG KONG', 'BARBADOS')
      AND UPPER(TRIM(US_OR_CANADA)) = 'CANADA'
),

Mapped_AUs AS (
    SELECT DISTINCT 
        TRIM(Assessable_Unit_ID) AS Mapping_ID_Or_Name,
        TRIM(TPRM_Units) AS TPRM_Units
    FROM hive_metastore.ra_adido_2025.third_party_unit_mapping
    WHERE Assessable_Unit_ID IS NOT NULL
      AND NULLIF(TRIM(TPRM_Units), '') IS NOT NULL
),

Third_Party_Risk_Data AS (
    SELECT 
        EngagementNumber,
        KPI_Number,
        final_rating,
        REPLACE(REPLACE(REPLACE(REPLACE(REPLACE(REPLACE(owning_lob, 
            'CAD PB - RESL, Everyday Banking, Savings & Investing', 'CAD PB - RESL[COMMA] Everyday Banking[COMMA] Savings & Investing'),
            'CAD PB - Card Payments, Loyalty, & Personal Lending', 'CAD PB - Card Payments[COMMA] Loyalty[COMMA] & Personal Lending'),
            'Transformation, Enablement, & Customer Experience', 'Transformation[COMMA] Enablement[COMMA] & Customer Experience'),
            'TECE - Strategy, Change, & Operational Excellence', 'TECE - Strategy[COMMA] Change[COMMA] & Operational Excellence'),
            'TECE - Enterprise Digital Strategy, Innovation, & Payments', 'TECE - Enterprise Digital Strategy[COMMA] Innovation[COMMA] & Payments'),
            'P&T - Integrations, Data and Payments', 'P&T - Integrations[COMMA] Data and Payments') AS safe_owning_lob,
        REPLACE(REPLACE(REPLACE(REPLACE(REPLACE(REPLACE(lob_using, 
            'CAD PB - RESL, Everyday Banking, Savings & Investing', 'CAD PB - RESL[COMMA] Everyday Banking[COMMA] Savings & Investing'),
            'CAD PB - Card Payments, Loyalty, & Personal Lending', 'CAD PB - Card Payments[COMMA] Loyalty[COMMA] & Personal Lending'),
            'Transformation, Enablement, & Customer Experience', 'Transformation[COMMA] Enablement[COMMA] & Customer Experience'),
            'TECE - Strategy, Change, & Operational Excellence', 'TECE - Strategy[COMMA] Change[COMMA] & Operational Excellence'),
            'TECE - Enterprise Digital Strategy, Innovation, & Payments', 'TECE - Enterprise Digital Strategy[COMMA] Innovation[COMMA] & Payments'),
            'P&T - Integrations, Data and Payments', 'P&T - Integrations[COMMA] Data and Payments') AS safe_lob_using
    FROM hive_metastore.ra_adido_2025.abac_request_paul_v3
    WHERE EngagementNumber IS NOT NULL
),

Flagged_Engagements AS (
    SELECT DISTINCT
        EngagementNumber,
        safe_owning_lob,
        safe_lob_using
    FROM Third_Party_Risk_Data
    WHERE TRIM(KPI_Number) = '8.2' 
      AND UPPER(TRIM(final_rating)) = 'YES'
),

Flattened_LOBs AS (
    SELECT 
        EngagementNumber, 
        REPLACE(TRIM(exploded_lob), '[COMMA]', ',') AS Expanded_LOB
    FROM Flagged_Engagements
    LATERAL VIEW explode(split(concat_ws(',', safe_owning_lob, safe_lob_using), ',')) AS exploded_lob
    WHERE exploded_lob IS NOT NULL AND TRIM(exploded_lob) != ''
),

Resolved_Rows AS (
    SELECT DISTINCT
        mast.BusinessID,
        mast.AU_Name,
        mast.Business_Segment,
        f.EngagementNumber,
        f.Expanded_LOB,
        map.TPRM_Units,
        map.Mapping_ID_Or_Name
    FROM Flattened_LOBs f
    INNER JOIN Mapped_AUs map
        ON UPPER(f.Expanded_LOB) LIKE '%' || UPPER(map.TPRM_Units) || '%'
    INNER JOIN Master_AUs mast
        ON UPPER(TRIM(map.Mapping_ID_Or_Name)) = UPPER(TRIM(mast.AU_Name))
        OR TRIM(map.Mapping_ID_Or_Name) = TRIM(mast.BusinessID)
),

Ranked_Resolved_Rows AS (
    SELECT 
        *,
        ROW_NUMBER() OVER (
            PARTITION BY BusinessID
            ORDER BY EngagementNumber, Expanded_LOB, TPRM_Units, Mapping_ID_Or_Name
        ) AS Debug_Row_Num
    FROM Resolved_Rows
)

-- Output columns:
-- BusinessID: Master AU ID from the ABAC AU anchor list.
-- AU_Name: Master AU name tied to BusinessID.
-- Business_Segment: Business segment carried from the master AU list.
-- EngagementNumber: Qualifying engagement number shown in the sample evidence set.
-- Expanded_LOB: Flattened LOB value that matched to the mapping table.
-- TPRM_Units: Mapping-table TPRM unit matched by Expanded_LOB.
-- Mapping_ID_Or_Name: Assessable-unit ID or name returned from the mapping table.
-- Bridge_Method: Indicates whether the bridge matched by AU ID, AU name, or a dual-bridge fallback.
-- Debug_Row_Num: Row number used to keep the sample output capped and readable.
SELECT 
    BusinessID,
    AU_Name,
    Business_Segment,
    EngagementNumber,
    Expanded_LOB,
    TPRM_Units,
    Mapping_ID_Or_Name,
    CASE 
        WHEN TRIM(Mapping_ID_Or_Name) = TRIM(BusinessID) THEN 'Matched on AU ID'
        WHEN UPPER(TRIM(Mapping_ID_Or_Name)) = UPPER(TRIM(AU_Name)) THEN 'Matched on AU Name'
        ELSE 'Matched through dual bridge'
    END AS Bridge_Method,
    Debug_Row_Num
FROM Ranked_Resolved_Rows
WHERE Debug_Row_Num <= 25
ORDER BY BusinessID, Debug_Row_Num;
